In [1]:
from pathlib import Path

import json
import time
import joblib
import numpy as np
import pandas as pd

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "heart_disease_clean.csv"
)

comparison_report_path = (
    project_root
    / "reports"
    / "model_comparison.csv"
)

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {processed_data_path}"
    )

if not comparison_report_path.exists():
    raise FileNotFoundError(
        f"Model comparison report not found: "
        f"{comparison_report_path}"
    )

df = pd.read_csv(processed_data_path)
comparison_results = pd.read_csv(
    comparison_report_path
)

target_column = "heart_disease"

if target_column not in df.columns:
    raise KeyError(
        f"Target column '{target_column}' was not found."
    )

required_result_columns = [
    "Model",
    "PR-AUC",
    "ROC-AUC",
    "Recall",
    "F1 Score"
]

missing_result_columns = [
    column
    for column in required_result_columns
    if column not in comparison_results.columns
]

if missing_result_columns:
    raise KeyError(
        f"Missing model-comparison columns: "
        f"{missing_result_columns}"
    )

if "Dataset" in comparison_results.columns:
    comparison_results = comparison_results[
        comparison_results["Dataset"] == "Testing"
    ].copy()

ranked_results = (
    comparison_results
    .sort_values(
        by=[
            "PR-AUC",
            "ROC-AUC",
            "Recall",
            "F1 Score"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

best_model_name = ranked_results.loc[0, "Model"]
best_model_result = ranked_results.iloc[0]

X = df.drop(columns=[target_column])
y = df[target_column].astype("int64")

print("Dataset and comparison report loaded successfully")
print("Dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Selected model:", best_model_name)
print(
    "Selected model PR-AUC:",
    round(float(best_model_result["PR-AUC"]), 4)
)
print(
    "Selected model ROC-AUC:",
    round(float(best_model_result["ROC-AUC"]), 4)
)

Dataset and comparison report loaded successfully
Dataset shape: (4238, 16)
Feature matrix shape: (4238, 15)
Selected model: Logistic Regression
Selected model PR-AUC: 0.2937
Selected model ROC-AUC: 0.6952


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

continuous_features = [
    "age",
    "cigarettes_per_day",
    "total_cholesterol",
    "systolic_bp",
    "diastolic_bp",
    "bmi",
    "heart_rate",
    "glucose"
]

binary_features = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

categorical_features = [
    "gender",
    "education"
]

all_defined_features = (
    continuous_features
    + binary_features
    + categorical_features
)

missing_features = sorted(
    set(X.columns) - set(all_defined_features)
)

unexpected_features = sorted(
    set(all_defined_features) - set(X.columns)
)

if missing_features:
    raise ValueError(
        f"Features not assigned to a group: {missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"Defined features not found: {unexpected_features}"
    )


def build_model_pipeline(model_name):
    continuous_steps = [
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]

    if model_name == "Logistic Regression":
        continuous_steps.append(
            (
                "scaler",
                StandardScaler()
            )
        )

    continuous_pipeline = Pipeline(
        steps=continuous_steps
    )

    binary_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "continuous",
                continuous_pipeline,
                continuous_features
            ),
            (
                "binary",
                binary_pipeline,
                binary_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    if model_name == "Logistic Regression":
        classifier = LogisticRegression(
            max_iter=1000,
            random_state=42
        )

    elif model_name == "Decision Tree":
        classifier = DecisionTreeClassifier(
            random_state=42
        )

    elif model_name == "Random Forest":
        classifier = RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )

    else:
        raise ValueError(
            f"Unsupported model: {model_name}"
        )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                classifier
            )
        ]
    )


final_pipeline = build_model_pipeline(
    best_model_name
)

print("Final model pipeline created successfully")
print("Selected classifier:", best_model_name)
print("Total input features:", len(all_defined_features))
print(final_pipeline)

Final model pipeline created successfully
Selected classifier: Logistic Regression
Total input features: 15
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('continuous',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'cigarettes_per_day',
                                                   'total_cholesterol',
                                                   'systolic_bp',
                                                   'diastolic_bp', 'bmi',
                                                   'heart_rate', 'glucose']),
                                                 ('binary',
                    

In [3]:
training_start = time.perf_counter()

final_pipeline.fit(
    X,
    y
)

training_end = time.perf_counter()
final_training_time = training_end - training_start

fitted_classifier = (
    final_pipeline
    .named_steps["classifier"]
)

fitted_preprocessor = (
    final_pipeline
    .named_steps["preprocessor"]
)

transformed_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

print("Final pipeline trained successfully")
print("Model:", best_model_name)
print("Training rows:", len(X))
print("Original input features:", X.shape[1])
print(
    "Transformed features:",
    len(transformed_feature_names)
)
print(
    f"Training time: {final_training_time:.4f} seconds"
)
print(
    "Model classes:",
    fitted_classifier.classes_.tolist()
)

Final pipeline trained successfully
Model: Logistic Regression
Training rows: 4238
Original input features: 15
Transformed features: 19
Training time: 0.0340 seconds
Model classes: [0, 1]


In [4]:
from datetime import datetime, timezone

models_directory = project_root / "models"

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

model_path = (
    models_directory
    / "heart_disease_prediction_pipeline.joblib"
)

metadata_path = (
    models_directory
    / "heart_disease_model_metadata.json"
)

joblib.dump(
    final_pipeline,
    model_path
)

model_metadata = {
    "model_version": "1.0.0",
    "selected_model": best_model_name,
    "target_column": target_column,
    "class_labels": {
        "0": "No heart disease",
        "1": "Heart disease"
    },
    "input_features": X.columns.tolist(),
    "training_rows": int(len(X)),
    "training_columns": int(X.shape[1]),
    "positive_class_rate": float(y.mean()),
    "trained_on_complete_dataset": True,
    "selection_metric": "PR-AUC",
    "selection_results": {
        "accuracy": float(
            best_model_result["Accuracy"]
        ),
        "precision": float(
            best_model_result["Precision"]
        ),
        "recall": float(
            best_model_result["Recall"]
        ),
        "f1_score": float(
            best_model_result["F1 Score"]
        ),
        "roc_auc": float(
            best_model_result["ROC-AUC"]
        ),
        "pr_auc": float(
            best_model_result["PR-AUC"]
        )
    },
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

metadata_path.write_text(
    json.dumps(
        model_metadata,
        indent=4
    ),
    encoding="utf-8"
)

model_size_mb = (
    model_path.stat().st_size
    / (1024 * 1024)
)

print("Model pipeline saved successfully")
print("Model path:", model_path)
print("Metadata path:", metadata_path)
print(f"Model size: {model_size_mb:.2f} MB")

Model pipeline saved successfully
Model path: c:\Users\User\OneDrive\Desktop\Heart-Disease-Classification\models\heart_disease_prediction_pipeline.joblib
Metadata path: c:\Users\User\OneDrive\Desktop\Heart-Disease-Classification\models\heart_disease_model_metadata.json
Model size: 0.01 MB


In [5]:
loaded_pipeline = joblib.load(
    model_path
)

validation_data = X.head(20).copy()

original_predictions = final_pipeline.predict(
    validation_data
)

loaded_predictions = loaded_pipeline.predict(
    validation_data
)

original_probabilities = final_pipeline.predict_proba(
    validation_data
)[:, 1]

loaded_probabilities = loaded_pipeline.predict_proba(
    validation_data
)[:, 1]

predictions_match = np.array_equal(
    original_predictions,
    loaded_predictions
)

probabilities_match = np.allclose(
    original_probabilities,
    loaded_probabilities
)

if not predictions_match:
    raise AssertionError(
        "Loaded model predictions do not match"
    )

if not probabilities_match:
    raise AssertionError(
        "Loaded model probabilities do not match"
    )

validation_summary = pd.DataFrame({
    "Original Prediction": original_predictions,
    "Loaded Prediction": loaded_predictions,
    "Original Probability": original_probabilities,
    "Loaded Probability": loaded_probabilities
})

print("Saved model validation completed successfully")
print("Predictions match:", predictions_match)
print("Probabilities match:", probabilities_match)
print(
    "Loaded pipeline steps:",
    list(loaded_pipeline.named_steps.keys())
)

display(validation_summary.head(10))

Saved model validation completed successfully
Predictions match: True
Probabilities match: True
Loaded pipeline steps: ['preprocessor', 'classifier']


,Original Prediction,Loaded Prediction,Original Probability,Loaded Probability
0,0,0,0.048279,0.048279
1,0,0,0.047453,0.047453
2,0,0,0.154841,0.154841
3,0,0,0.365076,0.365076
4,0,0,0.105365,0.105365
5,0,0,0.113781,0.113781
6,0,0,0.185482,0.185482
7,0,0,0.060280,0.060280
8,0,0,0.196479,0.196479
9,0,0,0.253545,0.253545
